# 🏥 Secure IoMT — ECG CNN-LSTM Training
**GPU-accelerated training on MIT-BIH Arrhythmia Dataset**

### Steps:
1. Run all cells top to bottom (`Runtime → Run all`)
2. Wait ~15–20 min for training
3. Download `ecg_cnn_lstm.keras` from the last cell
4. Put it in `d:\project\Python\models\` on your PC

In [ ]:
# ── Cell 1: Verify GPU is available ──────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    print('✅ GPU detected — training will be fast!')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install wfdb scipy scikit-learn -q
print('✅ All dependencies ready')

In [ ]:
# ── Cell 3: Signal preprocessing utilities ───────────────────────────────────
import numpy as np
from scipy.signal import butter, filtfilt

FS         = 360
WINDOW_LEN = 360
LOWCUT     = 0.5
HIGHCUT    = 40.0

def bandpass_filter(signal, fs=FS):
    nyq  = 0.5 * fs
    b, a = butter(4, [LOWCUT/nyq, HIGHCUT/nyq], btype='band')
    return filtfilt(b, a, signal)

def normalise(signal):
    mu, sigma = signal.mean(), signal.std()
    if sigma < 1e-6:
        return np.zeros_like(signal)
    return (signal - mu) / sigma

print('✅ Preprocessing utilities defined')

In [ ]:
# ── Cell 4: CNN-LSTM model architecture ──────────────────────────────────────
from tensorflow import keras

N_CLASSES  = 5
CLASS_NAMES = ['Normal', 'Tachycardia', 'Bradycardia', 'Arrhythmia', 'Noise']
MODEL_PATH = 'ecg_cnn_lstm.keras'

def build_model(input_len=WINDOW_LEN, n_classes=N_CLASSES):
    inputs = keras.Input(shape=(input_len, 1), name='ecg_input')
    x = keras.layers.Conv1D(32, 5, padding='same', activation='relu')(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Conv1D(64, 3, padding='same', activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling1D(2)(x)
    x = keras.layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling1D(2)(x)
    x = keras.layers.LSTM(64, return_sequences=False)(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(n_classes, activation='softmax', name='class_probs')(x)
    model = keras.Model(inputs, outputs, name='CNN_LSTM_ECG')
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()

In [ ]:
# ── Cell 5: Download & process MIT-BIH dataset ───────────────────────────────
# This cell downloads ~125 MB from PhysioNet (free, public)
import wfdb
import warnings
warnings.filterwarnings('ignore')

MIT_BIH_RECORDS = [
    '100','101','102','103','104','105','106','107','108','109',
    '111','112','113','114','115','116','117','118','119','121',
    '122','123','124','200','201','202','203','205','207','208',
    '209','210','212','213','214','215','217','219','220','221',
    '222','223','228','230','231','232','233','234'
]

NORMAL_SYMS = {'N', 'L', 'R', 'e', 'j'}
ARRHY_SYMS  = {'V', 'F', 'f', 'E', '/', 'A', 'a', 'J', 'S'}
STEP = FS

def classify_window(ann_samples, ann_symbols, w_start, w_end):
    in_win = [sym for s, sym in zip(ann_samples, ann_symbols)
              if w_start <= s < w_end]
    if not in_win:
        return 4  # Noise
    beats = [s for s, sym in zip(ann_samples, ann_symbols)
             if w_start <= s < w_end and sym in NORMAL_SYMS | ARRHY_SYMS]
    if len(beats) >= 2:
        avg_rr = np.mean(np.diff(beats))
        hr = 60.0 * FS / avg_rr if avg_rr > 0 else 75
    else:
        hr = 75
    if any(s in ARRHY_SYMS for s in in_win):
        return 3  # Arrhythmia
    if hr > 100:
        return 1  # Tachycardia
    if hr < 60:
        return 2  # Bradycardia
    return 0      # Normal

X, y = [], []
print('Downloading MIT-BIH records (~125 MB total)...')

for rec_name in MIT_BIH_RECORDS:
    try:
        record = wfdb.rdrecord(rec_name, pn_dir='mitdb')
        ann    = wfdb.rdann(rec_name, 'atr', pn_dir='mitdb')
    except Exception as e:
        print(f'  Skipping {rec_name}: {e}')
        continue

    signal = record.p_signal[:, 0].astype(np.float32)
    n_wins = (len(signal) - WINDOW_LEN) // STEP

    for i in range(n_wins):
        w_start = i * STEP
        w_end   = w_start + WINDOW_LEN
        window  = signal[w_start:w_end]
        try:
            window = bandpass_filter(window)
        except Exception:
            pass
        window = normalise(window)
        X.append(window)
        y.append(classify_window(ann.sample, ann.symbol, w_start, w_end))

    print(f'  {rec_name}: {n_wins} windows')

X = np.array(X, dtype=np.float32).reshape(-1, WINDOW_LEN, 1)
y = np.array(y, dtype=np.int32)

print(f'\n✅ Dataset ready: {X.shape[0]} windows')
for i, name in enumerate(CLASS_NAMES):
    print(f'   {i} {name}: {int(np.sum(y==i))}')

In [ ]:
# ── Cell 6: Train ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True, verbose=1)
]

print('🚀 Training started...')
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=30,
    batch_size=128,
    callbacks=callbacks
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'\n✅ Test accuracy: {acc*100:.2f}%  |  Test loss: {loss:.4f}')
model.save(MODEL_PATH)
print(f'✅ Model saved to {MODEL_PATH}')

In [ ]:
# ── Cell 7: Download the trained model to your PC ────────────────────────────
from google.colab import files
import os

if os.path.exists(MODEL_PATH):
    print(f'✅ Downloading {MODEL_PATH}...')
    files.download(MODEL_PATH)
    print('Done! Save it to  d:\\project\\Python\\models\\ecg_cnn_lstm.keras')
else:
    print('❌ Model file not found. Check if training completed successfully.')